# Quick SLM — 07 · SFT evaluation (Gemma 4 judge)

Same shape as the pretraining battery: **03** generates and caches, **06** scores
the cache with a single Gemma 4 judge. This notebook does both for the
fine-tuned checkpoint.

| | pretraining | here |
|---|---|---|
| cached generations | `logs/probe_outputs_<label>.json` | `logs/sft_probe_outputs_<label>.json` |
| judge scores | `logs/scored_outputs_<label>.json` | `logs/sft_scored_outputs_<label>.json` |

Every held-out response is scored 0–5 by Gemma 4 against the call a correct
response would make. There is no parser and no format gate: the first run of
this notebook scored 0/1386 because one brittle regex marked every generation
malformed, which said nothing about whether the calls were right. The judge
reads the response as written.

Generation and scoring are separate cells writing separate files, so the judge
can be re-run — a different rubric, a different judge — without paying for
generation again.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
# bitsandbytes provides the 4-bit quantization for the Gemma judge, as in 06.
!pip -q install --upgrade transformers accelerate safetensors tokenizers datasketch tqdm pandas bitsandbytes

## 3. Locate the repository and check framework support

In [ ]:
import sys, json, re
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/quick-slm/code')
framework_dir = REPO_DIR / 'framework'
assert framework_dir.is_dir(), (
    f'{framework_dir} not found. The tree moved from code/src to code/framework; '
    'upload framework/ and training/v1/ and delete the stale code/src.'
)
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))

from v1.quick_slm_trainer.support import require_framework
require_framework('v1', REPO_DIR)

from v1.quick_slm_trainer.paths import Layout
from v1.quick_slm_trainer.config import sft_v1

DRIVE_ROOT = Path('/content/drive/MyDrive/quick-slm')
LOGS_DIR   = DRIVE_ROOT / 'logs'
LOGS_DIR.mkdir(parents=True, exist_ok=True)

layout = Layout(drive_root=DRIVE_ROOT)
cfg = sft_v1()
print('framework OK; ctx', cfg.data.ctx)

## 4. Rebuild the held-out split

The split is derived, not stored, and the counts are checked against what was
packed. A mismatch means the model trained on what is about to be scored.

In [ ]:
from v1.quick_slm_trainer.sft import load_and_validate
from v1.quick_slm_trainer.sft.corpus import category_histogram, paired_integrity
from v1.quick_slm_trainer.sft.dedup import dedup_examples
from v1.quick_slm_trainer.sft.pack import split_examples

validated, val_stats, _ = load_and_validate(layout, cfg.sft)
deduped = dedup_examples(validated, threshold=cfg.sft.dedup_jaccard,
                         num_perm=cfg.sft.minhash_perms, progress=True)

corpus = list(deduped)          # 04b ran with REBALANCE = False
train_ex, val_ex = split_examples(corpus, val_fraction=cfg.sft.val_fraction)

recorded = json.loads(layout.sft_stats_path.read_text())

def _rec(*path):
    node = recorded
    for k in path:
        if not isinstance(node, dict) or k not in node:
            return None
        node = node[k]
    return node

checks = [
    ('corpus', len(corpus),   _rec('after_rebalance', 'examples') or _rec('after_dedup', 'examples')),
    ('train',  len(train_ex), _rec('pack', 'train', 'examples_in')),
    ('val',    len(val_ex),   _rec('pack', 'val', 'examples_in')),
]
known = [(n, got, want) for n, got, want in checks if want is not None]
assert known, f'corpus_stats.json records no counts (keys: {sorted(recorded)}). Re-run 04b with PACK = True.'
bad = [(n, got, want) for n, got, want in known if got != want]
assert not bad, (
    'the rebuilt corpus is NOT the one that was packed -- '
    + '; '.join(f'{n}: rebuilt {got:,}, pack recorded {want:,}' for n, got, want in bad)
    + '. Re-run 04b with PACK = True, then re-run 05.'
)
for n, got, _ in known:
    print(f'  {n:<7}{got:>8,}  matches the pack')

n_pairs, broken = paired_integrity(val_ex)
print(f'\nval counterfactual pairs {n_pairs}  broken {len(broken)}')
print('val by category:', category_histogram(val_ex))

## 5. Choose the checkpoints to score

**Both** the base checkpoint and the fine-tuned one, on the same held-out set,
scored by the same judge. That difference is the measurement, not the SFT number
alone: it is what separates "fine-tuning taught tool-calling" from "pretraining
had already taught it".

That distinction decides a v2 question directly. v1 spent 2.5 % of its
pretraining budget on function-calling data. If the base checkpoint scores near
zero here and the fine-tuned one does not, that budget bought nothing that
fine-tuning did not deliver anyway, and v2 is right to drop it.

Add step checkpoints to the list if you want the curve rather than the endpoints.

In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from v1.quick_slm_trainer.template import render_prompt

BATCH   = 32
MAX_NEW = 192

# (label, directory). The label names the cache file and the table row.
CHECKPOINTS = [
    ('base',      layout.final_dir),        # pretrained only, never fine-tuned
    ('sft-final', layout.sft_final_dir),    # after 05_sft_train
    # ('sft-step-899', layout.sft_ckpt_dir / 'step_0000899'),
]

def sft_probe_outputs_path(label):
    return LOGS_DIR / f'sft_probe_outputs_{label}.json'

def sft_scored_path(label):
    return LOGS_DIR / f'sft_scored_outputs_{label}.json'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if device == 'cuda' else torch.float32

tok = AutoTokenizer.from_pretrained(str(layout.tokenizer_dir))
tok.padding_side = 'left'   # right padding detaches the prompt from the continuation
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

print(f'{"label":<16}{"exists":>8}  directory')
print('-' * 74)
for label, d in CHECKPOINTS:
    print(f'{label:<16}{str(d.is_dir()):>8}  {d}')
missing = [l for l, d in CHECKPOINTS if not d.is_dir()]
assert not missing, f'no such checkpoint directory for: {missing}'

## 6. Generate, and cache the generations

Mirrors 03: one cache file per checkpoint, holding
`{label, hf_dir, outputs: [{prompt, expected, response, ...}]}`. Greedy — the
question is what the model believes, not what it can be sampled into.

Already-cached checkpoints are skipped, so adding a checkpoint to the list above
and re-running costs only that one.

In [ ]:
REGENERATE = False   # True -> re-generate even for checkpoints already cached

def expected_of(ex):
    '''The call a correct response makes. First assistant turn only -- later
    turns are conditioned on tool results the model never received.'''
    for turn in ex.turns:
        calls = getattr(turn, 'calls', None)
        if calls is not None:
            return json.dumps(list(calls), ensure_ascii=False)
    return '[]'

prompts = [render_prompt(e) for e in val_ex]

def generate_all(model):
    out = []
    for i in tqdm(range(0, len(prompts), BATCH), desc='  generate', leave=False):
        chunk = prompts[i:i + BATCH]
        enc = tok(chunk, return_tensors='pt', padding=True, truncation=True,
                  max_length=cfg.data.ctx - MAX_NEW).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        cut = enc['input_ids'].shape[1]
        for g in gen:
            # skip_special_tokens=False on purpose. <think>, </think> and
            # <response> are registered tokens, so skipping specials deletes the
            # entire envelope: the first run of this notebook produced 1,386
            # responses containing not one tag, which read as a total format
            # failure and was nothing of the kind.
            text = tok.decode(g[cut:], skip_special_tokens=False)
            for pad in (tok.pad_token or '', tok.eos_token or ''):
                if pad:
                    text = text.replace(pad, '')
            # The model has no stop criterion and does not halt after its call:
            # it invents tool results and keeps going, emitting 2.9 call arrays
            # per response on average. Cut at the first closed response so the
            # judge scores the answer rather than the rambling that follows it.
            end = text.find('</response>')
            out.append(text if end < 0 else text[:end + len('</response>')])
    return out

for label, ckpt in CHECKPOINTS:
    path = sft_probe_outputs_path(label)
    if path.exists() and not REGENERATE:
        print(f'skip {label} — already cached at {path.name}')
        continue

    print(f'\ngenerating {label} from {ckpt}')
    model = AutoModelForCausalLM.from_pretrained(str(ckpt), dtype=dtype).to(device).eval()
    responses = generate_all(model)
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()      # the judge needs the memory back

    path.write_text(json.dumps({
        'label': label,
        'hf_dir': str(ckpt),
        'outputs': [
            {'prompt': e.turns[0].text, 'expected': expected_of(e), 'response': r,
             'category': e.category, 'subtype': e.meta.get('subtype', ''),
             'spec_id': e.meta.get('spec_id'), 'group': e.meta.get('group'),
             'state': e.state}
            for e, r in zip(val_ex, responses)
        ],
    }, indent=2, default=str))
    print(f'  wrote {path}  ({len(responses):,} generations)')
    print(f'  first response: {responses[0][:200]!r}')

## 7. Load the Gemma 4 judge

The same model and the same 4-bit setup as 06, so an SFT score and a
pretraining score mean the same thing on the same scale.

The judge also wrote the SFT corpus. A model scoring text distributed like its
own output is biased toward flattering it, and that has to travel with any
number quoted from here.

In [ ]:
from transformers import BitsAndBytesConfig

GEMMA_MODEL = 'google/gemma-4-31B-it-qat-q4_0-unquantized'   # identical to 06

_quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
print(f'loading judge: {GEMMA_MODEL} (4-bit nf4)')
gemma_tok = AutoTokenizer.from_pretrained(GEMMA_MODEL)
gemma = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL, quantization_config=_quant_cfg, device_map='auto')
gemma.eval()
print(f'  loaded — {sum(p.numel() for p in gemma.parameters())/1e9:.1f} B params')

JUDGE_RUBRIC = """You are evaluating one output from a small (103M parameter) tool-calling model that has just been fine-tuned.

The model is given a user request and a set of tools. A correct response reasons briefly, then emits the tool call shown under EXPECTED — the same tool, with the same argument values. Judge the substance of the call, not its punctuation or whitespace.

Score the response from 0 to 5:
- 0: empty, repetition, or no tool call at all
- 1: a tool call, but unrelated to the request
- 2: plausible tool, wrong task; or the right tool with an argument the user never supplied
- 3: right tool, wrong argument value
- 4: right tool and right arguments, with a flaw in the reasoning or the formatting
- 5: the expected call, with reasoning that supports it

If a STATE block appears in the prompt it is authoritative and outranks any MEMORY block. A response that follows memory against a conflicting state is wrong, however fluent it reads.

Be strict. This is a 103M model and most responses should score 0-2.

PROMPT:
{prompt}

EXPECTED (the call a correct response makes):
{expected}

MODEL RESPONSE:
{response}

Respond with EXACTLY this format and nothing else:
SCORE: <integer 0-5>
REASON: <one short sentence>"""

_SCORE_RE  = re.compile(r'SCORE:\s*(\d+)', re.IGNORECASE)
_REASON_RE = re.compile(r'REASON:\s*(.+)', re.IGNORECASE)

@torch.no_grad()
def gemma_score(prompt, expected, response):
    '''One (prompt, expected, response) -> (score, reason). score = -1 if unparsed.

    -1 rather than 0: a judge that failed to answer the format is not the same
    event as a model that scored zero, and averaging the two together would move
    the mean every time the judge rambled.
    '''
    msg = JUDGE_RUBRIC.format(prompt=prompt[:800], expected=expected, response=response[:800])
    text = gemma_tok.apply_chat_template([{'role': 'user', 'content': msg}],
                                         tokenize=False, add_generation_prompt=True)
    enc = gemma_tok(text, return_tensors='pt').to(gemma.device)
    out = gemma.generate(**enc, max_new_tokens=64, do_sample=False,
                         pad_token_id=gemma_tok.eos_token_id)
    reply = gemma_tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    m = _SCORE_RE.search(reply)
    r = _REASON_RE.search(reply)
    score = int(m.group(1)) if m else -1
    if score > 5:
        score = -1
    return score, (r.group(1).strip().split('\n')[0] if r else reply.strip()[:160])

## 8. Score every cached generation

One judge, every checkpoint, so the rows of the table below are comparable to
each other and to the pretraining battery.

In [ ]:
ONLY_MISSING = True   # False -> re-score everything with the current rubric

ALL_SCORED = []
for label, _ in CHECKPOINTS:
    out_path = sft_scored_path(label)
    if ONLY_MISSING and out_path.exists():
        prev = json.loads(out_path.read_text())
        if prev.get('judge') == GEMMA_MODEL:
            print(f'skip {label} — already scored by {GEMMA_MODEL}')
            ALL_SCORED.append(prev)
            continue

    record = json.loads(sft_probe_outputs_path(label).read_text())
    print(f'\nscoring {label} ({len(record["outputs"]):,} responses)')
    scored = {'label': label, 'hf_dir': record['hf_dir'], 'judge': GEMMA_MODEL, 'outputs': []}
    for o in tqdm(record['outputs'], desc=f'  {label}', leave=False):
        score, reason = gemma_score(o['prompt'], o['expected'], o['response'])
        scored['outputs'].append({**o, 'score': score, 'reason': reason})
    out_path.write_text(json.dumps(scored, indent=2, default=str))
    ALL_SCORED.append(scored)
    unparsed = sum(1 for o in scored['outputs'] if o['score'] < 0)
    print(f'  wrote {out_path.name}   unparsed judge replies: {unparsed:,}')

print(f'\nscored {len(ALL_SCORED)} checkpoints with judge = {GEMMA_MODEL}')

## 9. The comparison table

Mean score per category, 0–5, on the same scale as the pretraining battery. Rows
are checkpoints in the order listed in section 5, so the base row is the
counterfactual for every number in the fine-tuned row.

The counterfactual category is reported twice, and the second number is the one
that matters. The mean says how good the answers are; **grounded** says how many
*pairs* scored 4 or better on **both** branches. A model that ignores `<state>`
answers one branch of every pair well and the other badly, so it can hold a
respectable mean with a grounded rate of zero. Unparsed judge replies are
excluded rather than counted as zero.

In [ ]:
import pandas as pd
from collections import defaultdict

def summarise(scored):
    outs = scored['outputs']
    ok = [o for o in outs if o['score'] >= 0]
    row = {'checkpoint': scored['label']}
    for cat in sorted({o['category'] for o in outs}):
        xs = [o['score'] for o in ok if o['category'] == cat]
        row[cat] = round(sum(xs) / len(xs), 2) if xs else None
    row['ALL'] = round(sum(o['score'] for o in ok) / len(ok), 2) if ok else None
    row['>=4'] = round(sum(o['score'] >= 4 for o in ok) / len(ok), 3) if ok else None
    row['unparsed'] = len(outs) - len(ok)
    return row

def grounded(scored):
    pairs = defaultdict(list)
    for o in scored['outputs']:
        if o['category'] == 'state_memory_conflict' and o.get('group') and o['score'] >= 0:
            pairs[o['group']].append(o['score'])
    complete = {g: ss for g, ss in pairs.items() if len(ss) == 2}
    both = sum(1 for ss in complete.values() if min(ss) >= 4)
    one = sum(1 for ss in complete.values() if min(ss) < 4 <= max(ss))
    return len(complete), both, one

df = pd.DataFrame([summarise(s) for s in ALL_SCORED])
print(df.to_string(index=False))

print('\ncounterfactual pairs (both branches >= 4 = grounded):')
print(f'  {"checkpoint":<16}{"pairs":>7}{"grounded":>10}{"rate":>8}{"one branch":>12}')
for s in ALL_SCORED:
    n, both, one = grounded(s)
    rate = f'{both/n:.1%}' if n else '   -'
    print(f'  {s["label"]:<16}{n:>7}{both:>10}{rate:>8}{one:>12}')

# The delta is the finding, so state it rather than leaving it to be read off.
if len(ALL_SCORED) >= 2:
    base, final = df.iloc[0], df.iloc[-1]
    print(f'\n{base["checkpoint"]} -> {final["checkpoint"]}:')
    for col in df.columns:
        if col in ('checkpoint', 'unparsed') or base[col] is None or final[col] is None:
            continue
        print(f'  {col:<24}{base[col]:>6}  ->{final[col]:>6}   {final[col] - base[col]:+.2f}')

print('\nworst-scoring examples from the final checkpoint:')
ok = [o for o in ALL_SCORED[-1]['outputs'] if o['score'] >= 0]
for o in sorted(ok, key=lambda o: o['score'])[:5]:
    print(f"\n  [{o['category']}] score {o['score']} — {o['reason']}")
    print(f"    expected: {o['expected'][:110]}")
    print(f"    got     : {o['response'][:110]!r}")